# 02 — Full-Data Weekly Aggregates (for product questions)

The project has **two data layers**, and choosing between them is a design decision worth explaining in interviews:

| Layer | Built in | Used for | Why |
|---|---|---|---|
| 10% customer sample, row-level | Notebook 01 | Discount → loyalty (customer questions) | Needs each customer's full history; 3M rows is laptop-friendly |
| **All 31M rows, aggregated by week** | **This notebook** | Product lifecycle, trend spread, reference prices | Per-article patterns need *all* sales; a 10% sample leaves most articles with too few sales |

We read the full transactions file in chunks and keep only small weekly summaries, so memory stays low.

**Outputs**
- `article_week.parquet`: units, sales value, min/max price per **article × channel × week**
- `trend_week.parquet`: units and sales value per **product type × colour × garment group × age band × week**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
FIGURES = Path("../reports/figures")
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

AGE_BINS = [0, 24, 34, 44, 54, 200]
AGE_LABELS = ["16-24", "25-34", "35-44", "45-54", "55+"]

## 1. Lookup tables: customer age band and article attributes

In [ ]:
cust = pd.read_csv(RAW / "customers.csv", usecols=["customer_id", "age"])
cust["age_band"] = pd.cut(cust["age"], bins=AGE_BINS, labels=False)  # 0..4, NaN if age missing
cust["age_band"] = cust["age_band"].fillna(-1).astype("int8")          # -1 = unknown
age_band_of = cust.set_index("customer_id")["age_band"]
print(f"Customers: {len(cust):,} | unknown age: {(cust['age_band'] == -1).mean():.1%}")
del cust

articles = pd.read_csv(RAW / "articles.csv", dtype={"article_id": "int32"})
attrs = articles.set_index("article_id")[["product_type_no", "colour_group_code", "garment_group_no"]]
print(f"Articles: {len(articles):,}")

## 2. Aggregate all transactions in chunks

The file is sorted by date, so groups rarely span two chunks; the final `groupby` merges any that do. Weeks start on **Monday**.

In [ ]:
ART_WEEK = PROCESSED / "article_week.parquet"
TREND_WEEK = PROCESSED / "trend_week.parquet"

if ART_WEEK.exists() and TREND_WEEK.exists():
    article_week = pd.read_parquet(ART_WEEK)
    trend_week = pd.read_parquet(TREND_WEEK)
    print("Loaded existing aggregates.")
else:
    aw_parts, tw_parts, total = [], [], 0
    reader = pd.read_csv(
        RAW / "transactions_train.csv",
        chunksize=2_000_000,
        parse_dates=["t_dat"],
        dtype={"article_id": "int32", "price": "float32", "sales_channel_id": "int8"},
    )
    for i, c in enumerate(reader, 1):
        total += len(c)
        c["week"] = c["t_dat"] - pd.to_timedelta(c["t_dat"].dt.dayofweek, unit="D")

        aw_parts.append(
            c.groupby(["article_id", "sales_channel_id", "week"])
             .agg(units=("price", "size"), sales_value=("price", "sum"),
                  min_price=("price", "min"), max_price=("price", "max"))
             .reset_index()
        )

        c["age_band"] = c["customer_id"].map(age_band_of).fillna(-1).astype("int8")
        c = c.join(attrs, on="article_id")
        tw_parts.append(
            c.groupby(["week", "product_type_no", "colour_group_code", "garment_group_no", "age_band"])
             .agg(units=("price", "size"), sales_value=("price", "sum"))
             .reset_index()
        )
        print(f"chunk {i}: {total:,} rows processed")

    article_week = (pd.concat(aw_parts)
        .groupby(["article_id", "sales_channel_id", "week"])
        .agg(units=("units", "sum"), sales_value=("sales_value", "sum"),
             min_price=("min_price", "min"), max_price=("max_price", "max"))
        .reset_index())
    trend_week = (pd.concat(tw_parts)
        .groupby(["week", "product_type_no", "colour_group_code", "garment_group_no", "age_band"])
        .agg(units=("units", "sum"), sales_value=("sales_value", "sum"))
        .reset_index())

    article_week["units"] = article_week["units"].astype("int32")
    trend_week["units"] = trend_week["units"].astype("int32")
    article_week.to_parquet(ART_WEEK, index=False)
    trend_week.to_parquet(TREND_WEEK, index=False)

    assert article_week["units"].sum() == total, "Units don't add up to total rows"
    assert trend_week["units"].sum() == total, "Units don't add up to total rows"
    print(f"Done. {total:,} transactions -> {len(article_week):,} article-week rows, {len(trend_week):,} trend rows")

print(f"Weeks: {article_week['week'].min().date()} to {article_week['week'].max().date()} "
      f"({article_week['week'].nunique()} weeks)")

## 3. Feasibility: product lifecycle (#4)

An article's **launch week** = its first week with sales in the data. Two problems to handle:
- **Left-censoring:** articles already on sale in Sep 2018 look like they "launched" then. We exclude articles whose first sale is in the first 8 weeks.
- **Right-censoring:** articles launched near the end haven't finished their life. We exclude articles launched in the last 12 weeks.

In [ ]:
first_week = article_week.groupby("article_id")["week"].min()
start, end = article_week["week"].min(), article_week["week"].max()
usable = first_week[(first_week >= start + pd.Timedelta(weeks=8)) &
                    (first_week <= end - pd.Timedelta(weeks=12))]

print(f"Articles with sales:            {len(first_week):,}")
print(f"Usable launches (not censored): {len(usable):,} ({len(usable)/len(first_week):.0%})")

first_week.value_counts().sort_index().plot(figsize=(11, 3.5), title="Articles launched per week (first sale)")
plt.axvline(start + pd.Timedelta(weeks=8), ls="--", c="grey")
plt.axvline(end - pd.Timedelta(weeks=12), ls="--", c="grey")
plt.ylabel("Articles")
plt.tight_layout()
plt.savefig(FIGURES / "02_launches_per_week.png", dpi=150)
plt.show()

In [ ]:
# Preview: average lifecycle curve per garment group (weekly units as % of the article's peak week)
aw = (article_week[article_week["article_id"].isin(usable.index)]
      .groupby(["article_id", "week"], as_index=False)["units"].sum())
aw["weeks_since_launch"] = ((aw["week"] - aw["article_id"].map(first_week)).dt.days // 7).astype(int)
aw["pct_of_peak"] = aw["units"] / aw.groupby("article_id")["units"].transform("max")

# Only articles with enough sales to have a meaningful curve
big = aw.groupby("article_id")["units"].sum()
aw = aw[aw["article_id"].isin(big[big >= 50].index)]
aw = aw.merge(articles[["article_id", "garment_group_name"]], on="article_id")
print(f"Articles in curve preview (50+ units): {aw['article_id'].nunique():,}")

top_groups = aw.groupby("garment_group_name")["article_id"].nunique().nlargest(5).index
(aw[aw["garment_group_name"].isin(top_groups) & (aw["weeks_since_launch"] <= 30)]
   .pivot_table(index="weeks_since_launch", columns="garment_group_name",
                values="pct_of_peak", aggfunc="mean")
   .plot(figsize=(10, 4), title="Average lifecycle curve, top 5 garment groups (preview)"))
plt.ylabel("Weekly units as share of peak")
plt.tight_layout()
plt.savefig(FIGURES / "02_lifecycle_preview.png", dpi=150)
plt.show()

Weeks with zero sales are missing rows here, which makes curves look higher than they are. In Week 3 we'll fill zero weeks properly; this is just a feasibility preview.

## 4. Feasibility: trend spread across age groups (#2)

In [ ]:
band_names = dict(enumerate(AGE_LABELS)) | {-1: "unknown"}
share = trend_week.groupby("age_band")["units"].sum()
share.index = share.index.map(band_names)
print("Share of all units by age band:")
print((share / share.sum()).round(3))
print()
print(f"Product types: {trend_week['product_type_no'].nunique()} | "
      f"colours: {trend_week['colour_group_code'].nunique()} | "
      f"garment groups: {trend_week['garment_group_no'].nunique()}")

In [ ]:
# Preview: for the 5 product types that grew most, when did each age band reach 50% of its total?
tw = trend_week[trend_week["age_band"] >= 0]
pt = tw.groupby(["product_type_no", "week"])["units"].sum().unstack(fill_value=0)
first_half = pt.loc[:, :pt.columns[len(pt.columns)//2 - 1]].sum(axis=1)
second_half = pt.loc[:, pt.columns[len(pt.columns)//2]:].sum(axis=1)
growth = (second_half / first_half.replace(0, np.nan)).where(pt.sum(axis=1) >= 500)
risers = growth.nlargest(5).index

pt_names = articles.drop_duplicates("product_type_no").set_index("product_type_no")["product_type_name"]
rows = []
for p in risers:
    sub = tw[tw["product_type_no"] == p].groupby(["age_band", "week"])["units"].sum()
    for band, s in sub.groupby(level=0):
        cum = s.droplevel(0).cumsum() / s.sum()
        rows.append({"product_type": f"{pt_names.get(p, p)} ({p})", "age_band": band_names[band],
                     "week_50pct": cum[cum >= 0.5].index[0].date()})
pd.DataFrame(rows).pivot(index="product_type", columns="age_band", values="week_50pct")

**How to read this:** for a product type that grew, if younger bands reach 50% of their purchases *earlier* than older bands, younger customers adopted it first. This is just a crude preview; Week 3 does it properly (weekly share, lag between age groups, many product types and colours).

## 5. What to write down
- Number of weeks and whether the first/last week are partial
- Usable launches for the lifecycle analysis
- Whether the preview lifecycle curves differ by garment group
- Age-band shares, and the unknown-age share
- Whether the trend preview hints at younger groups adopting earlier